# 07 · Final analysis: CNN, ablations, one-class models, telephone channel and cost rule

Completes RQ2–RQ4 for the final report and presentation, and re-exports the exploratory figures at print size.

* **Runtime:** use a **GPU runtime** (Runtime → Change runtime type → T4 GPU). About 1.5–2 hours, mostly unattended.
* **Needs:** the dataset in Drive (`audio-detection-human-vs-ai/raw/asvspoof2019_LA/LA.zip`) and, ideally, notebook 06's cache
  (`interim_cache/`). The data are copied to Colab's local disk; results go to `MyDrive/audio-detection-human-vs-ai/reports_final/`.
* Slow steps are cached in Drive (`final_cache/`), so if the session drops, **Run all** again and it resumes.
* The last cell downloads `reports_final.zip`.

In [ ]:
# --- Setup -------------------------------------------------------------------
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["VS_DATA_ROOT"] = "/content/vsdata"   # fast local disk
    if not os.path.exists("/content/audio-detection-human-vs-ai"):
        !git clone -q https://github.com/abhilash-u/audio-detection-human-vs-ai /content/audio-detection-human-vs-ai
    %cd /content/audio-detection-human-vs-ai
    !git pull -q
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import time, json, warnings, shutil, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import soundfile as sf
from src import config as C, data, preprocess as P, features as F, evaluation as E, models as M, sample_size as S
warnings.filterwarnings("ignore")
C.ensure_dirs()
DRIVE = Path("/content/drive/MyDrive/audio-detection-human-vs-ai") if IN_COLAB else C.DATA_ROOT
OUT = DRIVE / "reports_final"; ICACHE = DRIVE / "interim_cache"; FC = DRIVE / "final_cache"
for d in (OUT, ICACHE, FC): d.mkdir(parents=True, exist_ok=True)

# Print-ready figures: drawn at final size (max 6.5 in wide), 10 pt text, 300 dpi
plt.rcParams.update({"font.family": "serif", "font.size": 10, "axes.titlesize": 10, "axes.labelsize": 10,
                     "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9, "figure.dpi": 100,
                     "savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False})
HUMAN, SYN, KNOWN, GREY = "#2F4B66", "#C46A3A", "#9DB4CC", "#7F7F7F"
PAL = {"LR": "#7F7F7F", "RF": "#B07AA1", "XGBoost": "#59A14F", "SVM": HUMAN, "CNN": SYN, "IsolationForest": "#EDC948", "OneClassSVM": "#76B7B2"}
def save(fig, name):
    fig.savefig(OUT / f"{name}.png", bbox_inches="tight", facecolor="white"); plt.show()
def table(df, name):
    df.to_csv(OUT / f"{name}.csv", index=False); return df
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Local data:", C.DATA_ROOT, "| results:", OUT, "| device:", DEVICE)
if DEVICE == "cpu" and IN_COLAB:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then Run all (the CNN is very slow on CPU).")

## 1. Data: local copy, sample, clean and telephone audio

In [ ]:
# 1a. Copy the dataset to the local disk (about 5-10 min)
la = C.RAW_DIR / "LA"; local_zip = C.RAW_DIR / "LA.zip"; C.RAW_DIR.mkdir(parents=True, exist_ok=True)
drive_raw = DRIVE / "raw" / "asvspoof2019_LA"
t0 = time.time()
if not la.exists():
    if (drive_raw / "LA.zip").exists() and (drive_raw / "LA.zip").resolve() != local_zip.resolve():
        print("Copying LA.zip from Drive ..."); shutil.copy(drive_raw / "LA.zip", local_zip)
    elif not local_zip.exists():
        print("LA.zip not in Drive -> downloading ..."); data.download_la()
    if not la.exists():
        print("Unzipping ...")
        if subprocess.run(["unzip", "-q", "-o", str(local_zip), "-d", str(C.RAW_DIR)]).returncode != 0: data.download_la()
print(f"Dataset ready in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# 1b. The study sample (seed 42; reuses notebook 06's cache when present)
meta = data.build_metadata()
cached = ICACHE / "split_assignments.csv"
sample = pd.read_csv(cached) if cached.exists() else data.draw_sample(meta)
sample = sample.drop(columns=[c for c in ["path"] if c in sample.columns]).merge(meta[["clip_id", "path"]], on="clip_id")
sample["label_name"] = sample.label.map({0: "human", 1: "synthetic"})
sample["system_status"] = sample["system_status"].fillna("")
print("Study sample:", sample.shape, "| from cache:", cached.exists())

In [ ]:
# 1c. Clean (standardized) and telephone versions of every sampled clip, on local disk (about 10-15 min)
def process_one(row):
    cid = row["clip_id"]; cpath, ppath = C.CLEAN_DIR / f"{cid}.wav", C.PHONE_DIR / f"{cid}.wav"
    try:
        clip, info = P.standardise(P.load(row["path"]))
        if not cpath.exists(): sf.write(cpath, clip, C.SAMPLE_RATE, subtype="PCM_16")
        if not ppath.exists(): sf.write(ppath, P.telephone(clip), C.SAMPLE_RATE, subtype="PCM_16")
        return {"clip_id": cid, "read_error": "", **info}
    except Exception as e:
        return {"clip_id": cid, "read_error": str(e)[:120]}
t0 = time.time()
res = pd.DataFrame(Parallel(n_jobs=-1)(delayed(process_one)(r) for r in sample.to_dict("records")))
bad = res[res.read_error != ""]
print(f"Audio ready in {(time.time() - t0) / 60:.1f} min | unreadable: {len(bad)}")
sample = sample[~sample.clip_id.isin(bad.clip_id)].reset_index(drop=True)
for col in ["duration_raw_s", "leading_silence_s", "trailing_silence_s", "duration_trimmed_s"]:
    if col not in sample.columns:
        sample = sample.merge(res[["clip_id", col]], on="clip_id", how="left")

## 2. Features: 64 acoustic features (clean and telephone) and log-mel spectrograms

In [ ]:
import librosa
PITCH_HOP = 640   # 40 ms pYIN hop (same as notebook 06)
def extract_fast(path):
    y = P.load(path); sr = C.SAMPLE_RATE
    S_ = np.abs(librosa.stft(y, n_fft=C.N_FFT, hop_length=C.HOP_LENGTH, win_length=C.WIN_LENGTH))
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20, n_fft=C.N_FFT, hop_length=C.HOP_LENGTH, win_length=C.WIN_LENGTH)
    spec = [librosa.feature.spectral_centroid(S=S_, sr=sr)[0], librosa.feature.spectral_bandwidth(S=S_, sr=sr)[0],
            librosa.feature.spectral_rolloff(S=S_, sr=sr, roll_percent=0.85)[0], librosa.feature.spectral_flatness(S=S_)[0],
            librosa.feature.zero_crossing_rate(y, frame_length=C.WIN_LENGTH, hop_length=C.HOP_LENGTH)[0]]
    contrast = librosa.feature.spectral_contrast(S=S_, sr=sr, n_bands=6, fmin=100.0)
    rms = librosa.feature.rms(S=S_, frame_length=C.N_FFT)[0]; eps = 1e-10
    p5, p95 = np.percentile(rms, [5, 95])
    f0, voiced, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr, frame_length=2048, hop_length=PITCH_HOP)
    f0v = f0[voiced] if voiced is not None and voiced.any() else np.array([0.0])
    rms_db = 20 * np.log10(rms + eps)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1), np.ravel([[v.mean(), v.std()] for v in spec]), contrast.mean(axis=1),
                           [rms.mean(), rms.std(), 20 * np.log10((p95 + eps) / (p5 + eps))],
                           [np.nanmean(f0v), np.nanstd(f0v), float(np.mean(voiced)) if voiced is not None else 0.0,
                            float(np.mean(rms_db < rms_db.max() - 30))]]).astype(np.float32)

def feature_table(folder, cache_path, label):
    if cache_path.exists():
        X = pd.read_csv(cache_path)
        if set(X.clip_id) >= set(sample.clip_id):
            print(f"{label}: loaded from cache", X.shape); return X.set_index("clip_id").loc[sample.clip_id].reset_index()
    t0 = time.time()
    rows = Parallel(n_jobs=-1, verbose=2)(delayed(extract_fast)(folder / f"{c}.wav") for c in sample.clip_id)
    X = pd.DataFrame(np.vstack(rows), columns=F.FEATURES); X.insert(0, "clip_id", sample.clip_id.values)
    X.to_csv(cache_path, index=False); print(f"{label}: extracted {X.shape} in {(time.time() - t0) / 60:.1f} min")
    return X

Xc = feature_table(C.CLEAN_DIR, ICACHE / "features_clean.csv", "Clean features")          # from notebook 06 when available
Xp = feature_table(C.PHONE_DIR, FC / "features_telephone.csv", "Telephone features")        # about 25 min the first time
for X_ in (Xc, Xp):
    X_[F.FEATURES] = X_[F.FEATURES].replace([np.inf, -np.inf], np.nan)
med = Xc.loc[sample.split == "training", F.FEATURES].median()
Xc[F.FEATURES] = Xc[F.FEATURES].fillna(med); Xp[F.FEATURES] = Xp[F.FEATURES].fillna(med)
assert (Xc.clip_id.values == sample.clip_id.values).all() and (Xp.clip_id.values == sample.clip_id.values).all()

In [ ]:
# Log-mel spectrograms (128 x 401) for the CNN: clean for all clips, telephone for all clips (about 5 min)
def lm(path): return F.logmel(P.load(path))
t0 = time.time()
Lc = np.stack(Parallel(n_jobs=-1)(delayed(lm)(C.CLEAN_DIR / f"{c}.wav") for c in sample.clip_id))
Lp = np.stack(Parallel(n_jobs=-1)(delayed(lm)(C.PHONE_DIR / f"{c}.wav") for c in sample.clip_id))
print("Log-mels:", Lc.shape, Lp.shape, f"in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# Index sets used throughout
df = sample.reset_index(drop=True); y = df.label.values
idx = {s: np.where(df.split == s)[0] for s in df.split.unique()}
tr, va = idx["training"], idx["validation"]
tk = np.r_[idx["test_human"], idx["test_known"]]; tu = np.r_[idx["test_human"], idx["test_unknown"]]
te = np.r_[idx["test_human"], idx["test_known"], idx["test_unknown"]]
Fc, Fp = Xc[F.FEATURES], Xp[F.FEATURES]
SETS = {"Test-Known": tk, "Test-Unknown": tu}
print({k: len(v) for k, v in idx.items()})

## 3. Exploratory figures re-exported at print size (fixes legibility of the interim figures)

In [ ]:
comp = df.groupby(["split", "label_name"]).size().unstack(fill_value=0).reindex(["training", "validation", "test_human", "test_known", "test_unknown"]).fillna(0)
fig, ax = plt.subplots(figsize=(6.5, 2.6))
comp.plot.barh(stacked=True, color=[HUMAN, SYN], ax=ax, width=0.65)
ax.invert_yaxis(); ax.set_xlabel("Number of clips"); ax.set_ylabel(""); ax.legend(title="", frameon=False, loc="lower right")
for i, v in enumerate(comp.sum(axis=1).values): ax.text(v + 40, i, f"{int(v):,}", va="center", fontsize=9)
ax.set_xlim(0, comp.sum(axis=1).max() * 1.15)
save(fig, "E01_sample_composition")

fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.7))
for ax, col, lab in zip(axes, ["leading_silence_s", "duration_trimmed_s"], ["Leading silence (s)", "Speech after trimming (s)"]):
    for name, colr in [("human", HUMAN), ("synthetic", SYN)]:
        v = df.loc[df.label_name == name, col]
        ax.hist(v.clip(upper=v.quantile(0.995)), bins=35, alpha=0.6, color=colr, label=name, density=True)
    ax.set_xlabel(lab); ax.set_ylabel("Density")
axes[0].legend(frameon=False)
save(fig, "E02_silence_and_speech_by_class")

per_spk = df.groupby(["split", "speaker_id"]).size().rename("clips").reset_index()
order = ["training", "validation", "test_human", "test_known", "test_unknown"]
fig, ax = plt.subplots(figsize=(6.5, 2.6))
ax.boxplot([per_spk.loc[per_spk.split == s, "clips"] for s in order], vert=False)
ax.set_yticks(range(1, len(order) + 1)); ax.set_yticklabels(order); ax.invert_yaxis(); ax.set_xlabel("Clips per speaker")
save(fig, "E03_clips_per_speaker")

In [ ]:
pool = df[df.split.isin(["training", "validation"])].join(Fc)
res = E.feature_tests(pool, F.FEATURES)
rows = []
for f in res.feature.head(12):
    est, lo, hi = E.cluster_bootstrap(pool, lambda d, f=f: E.cohens_d(d.loc[d.label == 1, f].values, d.loc[d.label == 0, f].values), n_boot=500)
    rows.append({"feature": f, "ci_low": lo, "ci_high": hi})
top = res.head(12).merge(pd.DataFrame(rows), on="feature")
table(res.round(5), "R01_rq1_all_feature_tests"); table(top.round(4), "R02_rq1_top12_effects")
fig, ax = plt.subplots(figsize=(6.5, 4.0)); t = top.iloc[::-1]
ax.errorbar(t.cohens_d, range(len(t)), xerr=[t.cohens_d - t.ci_low, t.ci_high - t.cohens_d], fmt="o", color=HUMAN, ecolor=GREY, capsize=3)
ax.axvline(0, color="black", lw=0.8)
for v in (-0.8, -0.5, -0.2, 0.2, 0.5, 0.8): ax.axvline(v, color=GREY, lw=0.5, ls=":")
ax.set_yticks(range(len(t))); ax.set_yticklabels(t.feature); ax.set_xlabel("Cohen's d (synthetic − human), 95% speaker-cluster bootstrap CI")
save(fig, "E04_rq1_effect_sizes")

fig, axes = plt.subplots(2, 2, figsize=(6.5, 5.0))
for ax, f in zip(axes.ravel(), res.feature.head(4)):
    vp = ax.violinplot([pool.loc[pool.label == 0, f], pool.loc[pool.label == 1, f]], showmedians=True)
    for b, c in zip(vp["bodies"], [HUMAN, SYN]): b.set_facecolor(c); b.set_alpha(0.6)
    ax.set_xticks([1, 2]); ax.set_xticklabels(["human", "synthetic"]); ax.set_title(f)
fig.tight_layout(); save(fig, "E05_rq1_top4_distributions")

In [ ]:
trn = df.iloc[tr].join(Fc)
corr = trn[F.FEATURES].corr()
fig, ax = plt.subplots(figsize=(6.0, 5.2))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ticks = [10, 30, 45, 53, 58, 62]; labels = ["MFCC means", "MFCC SDs", "Spectral shape", "Contrast", "Energy", "Prosody"]
ax.set_xticks(ticks); ax.set_xticklabels(labels, rotation=35, ha="right"); ax.set_yticks(ticks); ax.set_yticklabels(labels)
for b in (20, 40, 50, 57, 60): ax.axhline(b - 0.5, color="white", lw=0.8); ax.axvline(b - 0.5, color="white", lw=0.8)
fig.colorbar(im, ax=ax, shrink=0.8, label="Pearson r")
save(fig, "E06_feature_correlation")

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
sc = StandardScaler().fit(Fc.iloc[tr]); pca = PCA(n_components=2, random_state=C.SEED).fit(sc.transform(Fc.iloc[tr])); ev = pca.explained_variance_ratio_
vis = df.iloc[np.r_[tr, tu]].sample(n=min(2500, len(tr) + len(tu)), random_state=C.SEED)
Pz = pca.transform(sc.transform(Fc.loc[vis.index]))
fig, axes = plt.subplots(1, 2, figsize=(6.5, 3.0), sharey=True)
for name, colr in [("human", HUMAN), ("synthetic", SYN)]:
    m = (vis.label_name == name).values; axes[0].scatter(Pz[m, 0], Pz[m, 1], s=3, alpha=0.35, c=colr, label=name)
grp = vis.system_status.map({"known": "known systems", "unseen": "unseen systems", "": "human"}).fillna("human")
for name, colr in [("human", HUMAN), ("known systems", KNOWN), ("unseen systems", SYN)]:
    m = (grp == name).values; axes[1].scatter(Pz[m, 0], Pz[m, 1], s=3, alpha=0.35, c=colr, label=name)
for ax in axes: ax.set_xlabel(f"PC1 ({ev[0]:.0%})"); ax.legend(frameon=False, markerscale=4, loc="upper right")
axes[0].set_ylabel(f"PC2 ({ev[1]:.0%})")
fig.tight_layout(); save(fig, "E07_pca")

bf = res.feature.iloc[0]
syn_all = df[df.label == 1].join(Fc)
st = syn_all.groupby(["attack_system_id", "system_status"])[bf].agg(["mean", "std"]).reset_index()
hm = df[df.label == 0].join(Fc)[bf].mean()
fig, ax = plt.subplots(figsize=(6.5, 3.0))
ax.bar(st.attack_system_id, st["mean"], yerr=st["std"], capsize=2, color=[KNOWN if s == "known" else SYN for s in st.system_status])
ax.axhline(hm, color=HUMAN, ls="--", lw=1.5, label="human mean"); ax.set_ylabel(bf); ax.legend(frameon=False)
ax.set_xlabel("Generation system (light = known, dark = unseen)"); ax.tick_params(axis="x", rotation=45)
save(fig, "E08_top_feature_by_system")

## 4. RQ2: classical models (3 seeds) and the lightweight CNN (3 seeds)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
def make(name, p, seed):
    est = {"LR": lambda: LogisticRegression(C=p["C"], max_iter=5000),
           "SVM": lambda: SVC(C=p["C"], gamma=p["gamma"], kernel="rbf", probability=True, random_state=seed),
           "RF": lambda: RandomForestClassifier(n_estimators=p["n"], max_depth=p["depth"], n_jobs=-1, random_state=seed),
           "XGBoost": lambda: XGBClassifier(n_estimators=p["n"], max_depth=p["depth"], learning_rate=p["lr"], subsample=0.8,
                                            colsample_bytree=0.8, eval_metric="logloss", random_state=seed, n_jobs=-1)}[name]()
    return M._pipe(est)

SEEDS = [42, 7, 2026]
best_params, scores, thr = {}, {}, {}
rows = []
for name in ["LR", "SVM", "RF", "XGBoost"]:
    _, params, _ = M.tune_classical(name, Fc.iloc[tr], y[tr], Fc.iloc[va], y[va]); best_params[name] = params
    for seed in (SEEDS if name != "LR" else SEEDS[:1]):
        t0 = time.time(); pipe = make(name, params, seed).fit(Fc.iloc[tr], y[tr]); secs = time.time() - t0
        th = E.eer(y[va], pipe.predict_proba(Fc.iloc[va])[:, 1])[1]
        for sn, r in SETS.items():
            s = pipe.predict_proba(Fc.iloc[r])[:, 1]
            rows.append({"model": name, "seed": seed, "set": sn, "train_s": round(secs, 1), **E.classification_report(y[r], s, th)})
            if seed == 42: scores[(name, "clean", sn)] = s
        if seed == 42: thr[name] = th; globals()[f"pipe_{name}"] = pipe
    print(name, params)
seed_tbl = table(pd.DataFrame(rows).round(4), "R03_classical_seeds")
summ = seed_tbl.groupby(["model", "set"]).agg(eer_mean=("eer", "mean"), eer_sd=("eer", "std"), auc_mean=("roc_auc", "mean")).reset_index()
table(summ.round(4), "R04_classical_seed_summary"); json.dump(best_params, open(OUT / "best_params.json", "w"), indent=1, default=str)
summ

In [ ]:
cnn_rows, cnn_runs = [], []
for seed in SEEDS:
    M.C.SEED = seed
    t0 = time.time()
    model, mu, sd, hist = M.train_cnn(Lc[tr], y[tr], Lc[va], y[va], epochs=25, batch=64, patience=5, device=DEVICE, verbose=False)
    secs = time.time() - t0
    s_va = M.predict_cnn(model, Lc[va], mu, sd); th = E.eer(y[va], s_va)[1]
    run = {"seed": seed, "model": model, "mu": mu, "sd": sd, "thr": th, "val_eer": E.eer(y[va], s_va)[0], "hist": hist, "secs": secs}
    cnn_runs.append(run)
    for sn, r in SETS.items():
        s = M.predict_cnn(model, Lc[r], mu, sd)
        cnn_rows.append({"model": "CNN", "seed": seed, "set": sn, "train_s": round(secs, 1), "epochs": len(hist), **E.classification_report(y[r], s, th)})
    print(f"CNN seed {seed}: val EER {run['val_eer']:.4f}, {len(hist)} epochs, {secs / 60:.1f} min")
M.C.SEED = 42
cnn_tbl = table(pd.DataFrame(cnn_rows).round(4), "R05_cnn_seeds")
best_run = min(cnn_runs, key=lambda r: r["val_eer"])
cnn, cmu, csd, thr["CNN"] = best_run["model"], best_run["mu"], best_run["sd"], best_run["thr"]
for sn, r in SETS.items(): scores[("CNN", "clean", sn)] = M.predict_cnn(cnn, Lc[r], cmu, csd)
torch.save({"state_dict": cnn.state_dict(), "mean": cmu, "std": csd, "threshold": thr["CNN"], "seed": best_run["seed"]}, FC / "cnn_best.pt")
fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.6))
for r_ in cnn_runs:
    h = pd.DataFrame(r_["hist"]); axes[0].plot(h.epoch, h.train_loss, label=f"seed {r_['seed']}"); axes[1].plot(h.epoch, h.val_eer, label=f"seed {r_['seed']}")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Training loss"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation EER")
axes[1].legend(frameon=False); fig.tight_layout(); save(fig, "G09_cnn_training_curves")
cnn_tbl.groupby("set")[["eer", "roc_auc"]].agg(["mean", "std"]).round(4)

In [ ]:
# RQ2 comparison: best classical (validation EER) vs CNN on identical clips; McNemar + speaker-cluster bootstrap CIs
cands = ["LR", "SVM", "RF", "XGBoost"]
val_eer = {n: E.eer(y[va], globals()[f"pipe_{n}"].predict_proba(Fc.iloc[va])[:, 1])[0] for n in cands}
BEST = min(val_eer, key=val_eer.get); print("Best classical model:", BEST, val_eer)
rows, mc_rows = [], []
for sn, r in SETS.items():
    for n in cands + ["CNN"]:
        rep_ = E.classification_report(y[r], scores[(n, "clean", sn)], thr[n])
        t = df.iloc[r][["speaker_id", "label"]].reset_index(drop=True).assign(s=scores[(n, "clean", sn)])
        e_, lo, hi = E.cluster_bootstrap(t, lambda d: E.eer(d.label.values, d.s.values)[0], n_boot=500)
        rows.append({"model": n, "set": sn, **rep_, "eer_ci_low": lo, "eer_ci_high": hi})
    a = (scores[("CNN", "clean", sn)] >= thr["CNN"]).astype(int) == y[r]; b = (scores[(BEST, "clean", sn)] >= thr[BEST]).astype(int) == y[r]
    mc_rows.append({"comparison": f"CNN vs {BEST}", "set": sn, "acc_cnn": a.mean(), f"acc_best": b.mean(), **E.mcnemar(a, b)})
rq2 = table(pd.DataFrame(rows).round(4), "R06_rq2_model_comparison"); table(pd.DataFrame(mc_rows).round(5), "R07_rq2_mcnemar")
display(pd.DataFrame(mc_rows).round(4))
rq2.pivot(index="model", columns="set", values=["eer", "roc_auc", "f1"]).round(3)

## 5. Ablation: does the detector still work without the 6.4–8 kHz cue?

In [ ]:
HIGHBAND = ["contrast_6_mean", "contrast_7_mean", "rolloff_mean", "rolloff_std", "centroid_mean", "centroid_std", "bandwidth_mean", "bandwidth_std"]
variants = {"All 64 features": F.FEATURES, "Without contrast_7_mean": [f for f in F.FEATURES if f != "contrast_7_mean"],
            "Without high-band features (8)": [f for f in F.FEATURES if f not in HIGHBAND]}
rows, abl_pipes = [], {}
for vname, cols in variants.items():
    pipe, params, v = M.tune_classical(BEST, Fc.iloc[tr][cols], y[tr], Fc.iloc[va][cols], y[va]); abl_pipes[vname] = (pipe, cols)
    th = E.eer(y[va], pipe.predict_proba(Fc.iloc[va][cols])[:, 1])[1]
    for sn, r in SETS.items():
        rows.append({"variant": vname, "n_features": len(cols), "set": sn, **E.classification_report(y[r], pipe.predict_proba(Fc.iloc[r][cols])[:, 1], th)})
abl = table(pd.DataFrame(rows).round(4), "R08_ablation")
from sklearn.inspection import permutation_importance
pipe_nc, cols_nc = abl_pipes["Without contrast_7_mean"]
pi_ = permutation_importance(pipe_nc, Fc.iloc[va][cols_nc], y[va], scoring="roc_auc", n_repeats=5, random_state=C.SEED, n_jobs=-1)
imp = pd.DataFrame({"feature": cols_nc, "importance": pi_.importances_mean, "sd": pi_.importances_std}).sort_values("importance", ascending=False)
table(imp.round(5), "R09_importance_without_contrast7")
fig, ax = plt.subplots(figsize=(6.5, 3.2)); t = imp.head(12).iloc[::-1]
ax.barh(t.feature, t.importance, xerr=t.sd, color=HUMAN); ax.set_xlabel(f"Drop in validation ROC-AUC when shuffled ({BEST}, without contrast_7_mean)")
save(fig, "G10_importance_without_contrast7")
abl.pivot(index="variant", columns="set", values=["eer", "roc_auc"]).round(3)

## 6. RQ3: one-class models trained on human speech only

In [ ]:
human_tr = tr[y[tr] == 0]; oc = {}
rows = []
for kind in ["IsolationForest", "OneClassSVM"]:
    pipe = M.fit_one_class(kind, Fc.iloc[human_tr]); th = E.eer(y[va], M.one_class_score(pipe, Fc.iloc[va]))[1]
    oc[kind] = (pipe, th); thr[kind] = th
    for sn, r in SETS.items():
        s = M.one_class_score(pipe, Fc.iloc[r]); scores[(kind, "clean", sn)] = s
        rows.append({"model": kind, "set": sn, **E.classification_report(y[r], s, th)})
ocr = table(pd.DataFrame(rows).round(4), "R10_rq3_one_class")
BEST_OC = ocr[ocr.set == "Test-Unknown"].sort_values("eer").model.iloc[0]
mc = []
for n in [BEST_OC, "CNN"]:
    a = (scores[(n, "clean", "Test-Unknown")] >= thr[n]).astype(int) == y[tu]; b = (scores[(BEST, "clean", "Test-Unknown")] >= thr[BEST]).astype(int) == y[tu]
    mc.append({"comparison": f"{n} vs {BEST}", "set": "Test-Unknown", "acc_a": a.mean(), "acc_best": b.mean(), **E.mcnemar(a, b)})
table(pd.DataFrame(mc).round(5), "R11_rq3_mcnemar")
# Detection rate per system for the best classical model, the CNN and the best one-class model
syn_idx = np.r_[idx["test_known"], idx["test_unknown"]]
det = df.iloc[syn_idx][["speaker_id", "attack_system_id", "attack_type", "system_status"]].reset_index(drop=True)
det_scores = {BEST: globals()[f"pipe_{BEST}"].predict_proba(Fc.iloc[syn_idx])[:, 1], "CNN": M.predict_cnn(cnn, Lc[syn_idx], cmu, csd),
              BEST_OC: M.one_class_score(oc[BEST_OC][0], Fc.iloc[syn_idx])}
rows = []
for n, s in det_scores.items():
    d_ = det.assign(detected=(s >= thr[n]).astype(int))
    for sid, g in d_.groupby("attack_system_id"):
        est, lo, hi = E.cluster_bootstrap(g, lambda d: d.detected.mean(), n_boot=300)
        rows.append({"model": n, "system": sid, "type": g.attack_type.iloc[0], "status": g.system_status.iloc[0], "n": len(g), "detection_rate": est, "ci_low": lo, "ci_high": hi})
per = table(pd.DataFrame(rows).round(4), "R12_rq3_detection_by_system")
order_sys = per[per.model == BEST].sort_values(["status", "detection_rate"]).system.tolist()
fig, ax = plt.subplots(figsize=(6.5, 3.2)); w = 0.27; xs = np.arange(len(order_sys))
for k, n in enumerate([BEST, "CNN", BEST_OC]):
    v = per[per.model == n].set_index("system").loc[order_sys]
    ax.bar(xs + (k - 1) * w, v.detection_rate, w, label=n, color=PAL.get(n, GREY))
ax.set_xticks(xs); ax.set_xticklabels([f"{s}\n({per[(per.model == BEST) & (per.system == s)].status.iloc[0][0]})" for s in order_sys], fontsize=8)
ax.set_ylabel("Detection rate"); ax.set_ylim(0, 1.05); ax.legend(frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.0))
save(fig, "G03_detection_by_system")
ocr.pivot(index="model", columns="set", values=["eer", "roc_auc", "fpr"]).round(3)

## 7. RQ4: telephone channel — degradation and retraining on telephone audio

In [ ]:
rows = []
pipe_best = globals()[f"pipe_{BEST}"]
for sn, r in SETS.items():
    s_clean = scores[(BEST, "clean", sn)]; s_phone = pipe_best.predict_proba(Fp.iloc[r])[:, 1]; scores[(BEST, "phone", sn)] = s_phone
    c_clean = M.predict_cnn(cnn, Lp[r], cmu, csd); scores[("CNN", "phone", sn)] = c_clean
    rows += [{"model": BEST, "train": "clean", "test": "clean", "set": sn, **E.classification_report(y[r], s_clean, thr[BEST])},
             {"model": BEST, "train": "clean", "test": "telephone", "set": sn, **E.classification_report(y[r], s_phone, thr[BEST])},
             {"model": "CNN", "train": "clean", "test": "clean", "set": sn, **E.classification_report(y[r], scores[("CNN", "clean", sn)], thr["CNN"])},
             {"model": "CNN", "train": "clean", "test": "telephone", "set": sn, **E.classification_report(y[r], c_clean, thr["CNN"])}]
# Retrain on telephone audio (band-limited training); thresholds from telephone validation
pipe_ph, params_ph, _ = M.tune_classical(BEST, Fp.iloc[tr], y[tr], Fp.iloc[va], y[va]); thr["phone"] = E.eer(y[va], pipe_ph.predict_proba(Fp.iloc[va])[:, 1])[1]
M.C.SEED = 42
cnn_ph, pmu, psd, _ = M.train_cnn(Lp[tr], y[tr], Lp[va], y[va], epochs=25, batch=64, patience=5, device=DEVICE, verbose=False)
thr["cnn_phone"] = E.eer(y[va], M.predict_cnn(cnn_ph, Lp[va], pmu, psd))[1]
for sn, r in SETS.items():
    s = pipe_ph.predict_proba(Fp.iloc[r])[:, 1]; scores[(BEST + "_phone", "phone", sn)] = s
    c = M.predict_cnn(cnn_ph, Lp[r], pmu, psd); scores[("CNN_phone", "phone", sn)] = c
    rows += [{"model": BEST, "train": "telephone", "test": "telephone", "set": sn, **E.classification_report(y[r], s, thr["phone"])},
             {"model": "CNN", "train": "telephone", "test": "telephone", "set": sn, **E.classification_report(y[r], c, thr["cnn_phone"])}]
tel = table(pd.DataFrame(rows).round(4), "R13_rq4_telephone")
# Paired McNemar: clean vs telephone audio, same model and threshold, all test clips
mc = []
s_c = pipe_best.predict_proba(Fc.iloc[te])[:, 1]; s_p = pipe_best.predict_proba(Fp.iloc[te])[:, 1]
mc.append({"model": BEST, "comparison": "clean vs telephone (clean-trained)", **E.mcnemar((s_c >= thr[BEST]).astype(int) == y[te], (s_p >= thr[BEST]).astype(int) == y[te])})
c_c = M.predict_cnn(cnn, Lc[te], cmu, csd); c_p = M.predict_cnn(cnn, Lp[te], cmu, csd)
mc.append({"model": "CNN", "comparison": "clean vs telephone (clean-trained)", **E.mcnemar((c_c >= thr["CNN"]).astype(int) == y[te], (c_p >= thr["CNN"]).astype(int) == y[te])})
table(pd.DataFrame(mc).round(5), "R14_rq4_mcnemar")
tel["scenario"] = tel.train + " → " + tel.test
fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.9), sharey=True)
for ax, sn in zip(axes, SETS):
    t = tel[tel.set == sn]; scen = ["clean → clean", "clean → telephone", "telephone → telephone"]; xs = np.arange(3); w = 0.38
    for k, n in enumerate([BEST, "CNN"]):
        v = t[t.model == n].set_index("scenario").reindex(scen).eer
        ax.bar(xs + (k - 0.5) * w, v, w, color=PAL[n] if n in PAL else HUMAN, label=n)
    ax.set_xticks(xs); ax.set_xticklabels(["clean→clean", "clean→phone", "phone→phone"], fontsize=8.5); ax.set_title(sn)
axes[0].set_ylabel("Equal error rate"); axes[0].legend(frameon=False)
fig.tight_layout(); save(fig, "G05_telephone_eer")
tel.pivot_table(index=["model", "scenario"], columns="set", values="eer").round(3)

## 8. RQ4: cost-optimal thresholds, sensitivity and the three-band risk rule (telephone model)

In [ ]:
# Deployment model = best of the telephone-trained models on telephone validation EER
cand = {BEST + " (telephone-trained)": (lambda r: pipe_ph.predict_proba(Fp.iloc[r])[:, 1]),
        "CNN (telephone-trained)": (lambda r: M.predict_cnn(cnn_ph, Lp[r], pmu, psd))}
DEP = min(cand, key=lambda k: E.eer(y[va], cand[k](va))[0]); score_fn = cand[DEP]; print("Deployment model:", DEP)
s_val = score_fn(va); base = {"c_fn": C.COST_FN, "c_fp": C.COST_FP, "c_review": C.COST_REVIEW}
rows = []
for pi_ in C.PREVALENCES:
    opt = E.cost_optimal_threshold(y[va], s_val, pi_)
    for sn, r in {**SETS, "All test": te}.items():
        rep_ = E.classification_report(y[r], score_fn(r), opt["threshold"])
        rows.append({"model": DEP, "prevalence": pi_, "set": sn, "threshold": opt["threshold"], "tpr": rep_["recall_tpr"], "fpr": rep_["fpr"],
                     "precision_at_prevalence": E.precision_at_prevalence(rep_["recall_tpr"], rep_["fpr"], pi_),
                     "expected_cost_per_1000": E.expected_cost(rep_["recall_tpr"], rep_["fpr"], pi_), "cost_no_screening": 1000 * pi_ * C.COST_FN})
cost = table(pd.DataFrame(rows).round(4), "R15_rq4_cost_thresholds")
sens = []
for pi_ in C.PREVALENCES:
    for key in base:
        for fct in (0.5, 1.0, 1.5):
            cs = {**base, key: base[key] * fct}; opt = E.cost_optimal_threshold(y[va], s_val, pi_, **cs)
            rep_ = E.classification_report(y[te], score_fn(te), opt["threshold"])
            sens.append({"prevalence": pi_, "varied": key, "factor": fct, "threshold": opt["threshold"],
                         "test_expected_cost": E.expected_cost(rep_["recall_tpr"], rep_["fpr"], pi_, **cs)})
sens = table(pd.DataFrame(sens).round(4), "R16_rq4_sensitivity")
s1 = sens[sens.prevalence == 0.01]; basec = s1[(s1.factor == 1.0)].test_expected_cost.iloc[0]
fig, ax = plt.subplots(figsize=(6.5, 2.4)); labs = {"c_fn": "Missed-fraud cost", "c_fp": "False-alarm cost", "c_review": "Review cost"}
for i, key in enumerate(base):
    lo_, hi_ = s1[(s1.varied == key) & (s1.factor == 0.5)].test_expected_cost.iloc[0], s1[(s1.varied == key) & (s1.factor == 1.5)].test_expected_cost.iloc[0]
    ax.barh(i, hi_ - basec, left=basec, color=SYN); ax.barh(i, lo_ - basec, left=basec, color=HUMAN)
ax.axvline(basec, color="black", lw=0.8); ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}")); ax.set_yticks(range(3)); ax.set_yticklabels([labs[k] for k in base])
ax.set_xlabel("Expected cost per 1,000 calls at 1% prevalence (₹); dark = −50%, orange = +50%")
save(fig, "G07_cost_sensitivity")
fig, ax = plt.subplots(figsize=(6.5, 2.8))
for pi_, ls in zip(C.PREVALENCES, ["-", "--"]):
    cc = E.cost_curve(y[va], s_val, pi_); ax.plot(cc.threshold.clip(0, 1), cc.expected_cost, ls, color=HUMAN, label=f"prevalence {pi_:.0%}")
ax.set_yscale("log"); ax.set_xlabel("Decision threshold on P(synthetic)"); ax.set_ylabel("Expected cost per 1,000 calls (₹, log)"); ax.legend(frameon=False)
save(fig, "G06_cost_curves")
# Risk bands: review = cost-optimal at 5% prevalence; high = cost-optimal at 1% (or review + 0.1)
t_rev = float(cost[(cost.prevalence == 0.05)].threshold.iloc[0]); t_high = float(max(cost[(cost.prevalence == 0.01)].threshold.iloc[0], min(t_rev + 0.1, 0.99)))
s_te = score_fn(te); band = np.where(s_te >= t_high, "High", np.where(s_te >= t_rev, "Review", "Low"))
grp = np.where(y[te] == 0, "Human", np.where(df.iloc[te].system_status.values == "known", "Synthetic (known)", "Synthetic (unseen)"))
bands = pd.crosstab(pd.Series(grp, name="group"), pd.Series(band, name="band"), normalize="index").reindex(columns=["Low", "Review", "High"]).fillna(0)
table(bands.round(4).reset_index(), "R17_risk_bands"); json.dump({"model": DEP, "review": t_rev, "high": t_high}, open(OUT / "risk_band_thresholds.json", "w"), indent=1)
fig, ax = plt.subplots(figsize=(6.5, 2.4)); left = np.zeros(len(bands))
for b_, colr in zip(["Low", "Review", "High"], ["#BFD3E6", "#EDC948", SYN]):
    ax.barh(bands.index, bands[b_], left=left, color=colr, label=b_); left += bands[b_].values
ax.set_xlabel("Share of clips"); ax.set_xlim(0, 1); ax.legend(frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.0)); ax.invert_yaxis()
save(fig, "G08_risk_bands")
cost

## 9. Summary figures, correctness ICC and per-clip predictions

In [ ]:
# EER by model, known vs unseen (clean audio)
models_ = ["LR", "RF", "XGBoost", "SVM", "CNN", "IsolationForest", "OneClassSVM"]
summ_rows = []
for n in models_:
    for sn, r in SETS.items():
        summ_rows.append({"model": n, "set": sn, "eer": E.eer(y[r], scores[(n, "clean", sn)])[0], "roc_auc": E.roc_auc_score(y[r], scores[(n, "clean", sn)])})
sm = table(pd.DataFrame(summ_rows).round(4), "R18_eer_summary")
fig, ax = plt.subplots(figsize=(6.5, 3.0)); xs = np.arange(len(models_)); w = 0.38
for k, (sn, colr) in enumerate([("Test-Known", KNOWN), ("Test-Unknown", SYN)]):
    v = sm[sm.set == sn].set_index("model").loc[models_].eer; ax.bar(xs + (k - 0.5) * w, v, w, color=colr, label=sn)
    for x_, v_ in zip(xs + (k - 0.5) * w, v): ax.text(x_, v_ + 0.008, f"{v_:.0%}", ha="center", fontsize=7.5)
ax.set_xticks(xs); ax.set_xticklabels(["LR", "RF", "XGBoost", "SVM", "CNN", "Isol. Forest", "1-class SVM"], fontsize=9)
ax.set_ylabel("Equal error rate"); ax.legend(frameon=False)
save(fig, "G01_eer_by_model")
from sklearn.metrics import roc_curve
fig, axes = plt.subplots(1, 2, figsize=(6.5, 3.0), sharey=True)
for ax, sn in zip(axes, SETS):
    r = SETS[sn]
    for n in ["SVM", "XGBoost", "CNN", BEST_OC]:
        f_, t_, _ = roc_curve(y[r], scores[(n, "clean", sn)]); ax.plot(f_, t_, color=PAL.get(n, GREY), label=n)
    ax.plot([0, 1], [0, 1], color=GREY, lw=0.6, ls=":"); ax.set_title(sn); ax.set_xlabel("False-positive rate")
axes[0].set_ylabel("True-positive rate"); axes[1].legend(frameon=False, loc="lower right")
fig.tight_layout(); save(fig, "G02_roc_curves")
# Intra-class correlation of correctness (speaker clustering of errors) -> design effect for RQ2/RQ4
rows = []
for sn, r in SETS.items():
    corr_ = ((scores[(BEST, "clean", sn)] >= thr[BEST]).astype(int) == y[r]).astype(float)
    spk = df.iloc[r].speaker_id.values; rho_c = max(S.estimate_icc(corr_, spk), 0.0)
    req = S.rq2(rho=rho_c)
    rows.append({"set": sn, "rho_correctness": rho_c, "deff": req["deff"], "n0_pairs": req["n0_pairs"],
                 "required_pairs": req["minimum_N"], "available_pairs": len(r), "meets": len(r) >= req["minimum_N"]})
table(pd.DataFrame(rows).round(4), "R19_icc_correctness")
pred = df.iloc[te][["clip_id", "speaker_id", "split", "label", "attack_system_id", "system_status"]].reset_index(drop=True)
for (n, ch, sn), s in scores.items():
    r = SETS[sn]; col = f"{n}_{ch}"
    if col not in pred: pred[col] = np.nan
    pos = pd.Index(df.iloc[te].clip_id).get_indexer(df.iloc[r].clip_id); pred.loc[pos, col] = s
table(pred, "R20_test_predictions")
pd.DataFrame(rows).round(3)

In [ ]:
summary = {"best_classical": BEST, "best_one_class": BEST_OC, "deployment_model": DEP, "cnn_best_seed": best_run["seed"],
           "risk_bands": {"review": t_rev, "high": t_high}, "params": best_params}
json.dump(summary, open(OUT / "summary.json", "w"), indent=1, default=str)
zip_path = shutil.make_archive(str(DRIVE / "reports_final"), "zip", OUT)
print("Saved", zip_path, "|", len(list(OUT.iterdir())), "files")
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)